# Entrenamiento de Modelo de Inventario (MobileNetV2)
Este notebook contiene el pipeline de entrenamiento para la detección de herramientas y útiles en el laboratorio.

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## 1. Carga de Dataset y Aumentación Realista
Se cargan las imágenes desde el directorio `../dataset` y se aplican aumentaciones que simulan las condiciones del laboratorio (sin flip vertical).

In [2]:
dataset_dir = '../dataset'
batch_size = 32
img_height = 224
img_width = 224

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True, # Permitido
    vertical_flip=False   # PROHIBIDO explícitamente
)

val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = val_datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

labels = (train_generator.class_indices)
labels = dict((v,k) for k,v in labels.items())
print("Etiquetas encontradas:", labels)

with open('../modelos_exportados/labels.txt', 'w') as f:
    for i in range(len(labels)):
        f.write(f"{labels[i]}\n")


Found 11120 images belonging to 3 classes.
Found 2779 images belonging to 3 classes.
Etiquetas encontradas: {0: 'clase1_tecnologia', 1: 'clase2_utiles', 2: 'clase3_vacio'}


## 2. Construcción del Modelo con Preprocesamiento Integrado (CRÍTICO)
Inyectamos una capa Lambda para normalizar de 0-255 a -1.0 - 1.0 dentro del modelo, eliminando la matemática en el dispositivo Edge.

In [3]:
num_classes = len(labels)

# 1. Capa de Entrada (UINT8 de 0-255 esperado en la app)
inputs = tf.keras.Input(shape=(img_height, img_width, 3), dtype=tf.uint8, name='image_input')

# 2 y 3. Preprocesamiento Inyectado Keras 3 (Cast y Normalización)
x = tf.keras.layers.Lambda(lambda t: (tf.cast(t, tf.float32) / 127.5) - 1.0)(inputs)

# 4. Cargar MobileNetV2 base
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # Congelar base

# 5. Cabeza de Clasificación
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='prediction')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prediction (Dense)              │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 3. Entrenamiento (Transfer Learning)

In [4]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# ENTRENAMIENTO REAL — 10 épocas con los datos cargados arriba
epochs = 10
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs
)

# Guardar backup del modelo entrenado
model.save('../modelos_exportados/modelo_base.h5')
print(f"Precisión final de validación: {history.history['val_accuracy'][-1]:.2%}")

Epoch 1/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 338s 963ms/step - accuracy: 0.6683 - loss: 0.6975 - val_accuracy: 0.7024 - val_loss: 0.9876
Epoch 2/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 334s 960ms/step - accuracy: 0.6833 - loss: 0.6358 - val_accuracy: 0.7136 - val_loss: 0.9075
Epoch 3/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 394s 1s/step - accuracy: 0.6953 - loss: 0.6195 - val_accuracy: 0.6358 - val_loss: 0.9181
Epoch 4/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 293s 843ms/step - accuracy: 0.6960 - loss: 0.6096 - val_accuracy: 0.7067 - val_loss: 0.9512
Epoch 5/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 299s 861ms/step - accuracy: 0.6932 - loss: 0.6111 - val_accuracy: 0.7110 - val_loss: 0.8705
Epoch 6/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 282s 811ms/step - accuracy: 0.6974 - loss: 0.6033 - val_accuracy: 0.7128 - val_loss: 0.9618
Epoch 7/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 290s 834ms/step - accuracy: 0.6973 - loss: 0.6038 - val_accuracy: 0.7136 - val_loss: 0.9531
Epoch 8/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 279s 801ms/step - accuracy: 0.6985 - lo

Precisión final de validación: 71.79%


## 4. Full Integer Quantization (INT8)
Cuantización del modelo a INT8 usando un Representative Dataset para calibrar las activaciones.

In [5]:
def representative_data_gen():
    for input_value, _ in train_generator:
        yield [input_value.astype(np.uint8)] # Inyectamos como uint8 crudo, tal como espera el modelo.
        break 

# Convertir el modelo
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# Asegurar que los tensores de entrada/salida permanezcan en formatos óptimos
# La entrada es explícitamente tf.uint8 (matrices de CameraX) y la salida tf.uint8/float.
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8  
converter.inference_output_type = tf.uint8 

try:
    tflite_quant_model = converter.convert()
    with open('../modelos_exportados/modelo_int8.tflite', 'wb') as f:
        f.write(tflite_quant_model)
    print("Modelo cuantizado guardado en ../modelos_exportados/modelo_int8.tflite")
    print(f"Tamaño del modelo: {os.path.getsize('../modelos_exportados/modelo_int8.tflite') / (1024 * 1024):.2f} MB")
except Exception as e:
    print("Error en la conversión (posiblemente porque no hay dataset aún):", e)


INFO:tensorflow:Assets written to: C:\Users\Alumno\AppData\Local\Temp\tmpqnidtvi0\assets


INFO:tensorflow:Assets written to: C:\Users\Alumno\AppData\Local\Temp\tmpqnidtvi0\assets


Saved artifact at 'C:\Users\Alumno\AppData\Local\Temp\tmpqnidtvi0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8, name='image_input')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2903305045008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2903305046352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2903305047120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2903305046736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2903305046928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2903305047312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2903305047696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2903305047504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2903305047888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2903305048080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2903305

c:\Users\Alumno\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Modelo cuantizado guardado en ../modelos_exportados/modelo_int8.tflite
Tamaño del modelo: 2.59 MB
